# Retail Orders Data Analysis

## Project Overview

This project analyzes retail order data to understand sales, profitability, and product performance.

The analysis includes data loading, data cleaning, transformation, SQL analysis, and business insights.

## 1. Environment Setup

Verify that the notebook is running in the intended Python environment
and confirm that the required libraries are available.

In [1]:
import sys
print(sys.executable)

C:\Users\Imraan\anaconda3\envs\data-analytics\python.exe


In [2]:
import pandas
import sqlalchemy

In [3]:
print("Pandas: ", pandas.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)

Pandas:  3.0.6
SQLAlchemy: 2.1.1


## 2. Install Required Packages

Install the packages needed to download the dataset and connect Python to SQL Server.

In [4]:
%pip install kaggle pyodbc

Note: you may need to restart the kernel to use updated packages.


In [5]:
import kaggle
import pyodbc

print("Kaggle: OK")
print("PyODBC: OK")

Kaggle: OK
PyODBC: OK


In [6]:
!kaggle auth login

You are already logged-in to Kaggle as [minhazulislam1999].
Please use the --force flag to override.


In [7]:
!kaggle datasets list -s retail-orders

ref                                                   title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
----------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
gabrielsantello/wholesale-and-retail-orders-dataset   Wholesale & Retail Orders Dataset                      3238262  2023-09-19 09:31:40.273000           4883         52                1  
ankitbansal06/retail-orders                           Retail Orders                                           205161  2024-04-13 08:31:36.377000          30107         48       0.47058824  
adarsh0806/influencer-merchandise-sales               Global Retail Sales Data: Orders, Reviews & Trends      125403  2024-12-10 14:07:52                  4096         38                1  
abbas829/ecommerce-sales-dataset                  

### Download the Dataset

Download the retail orders dataset from Kaggle for this project.

In [8]:
!kaggle datasets download ankitbansal06/retail-orders -f orders.csv

Dataset URL: https://www.kaggle.com/datasets/ankitbansal06/retail-orders
License(s): CC0-1.0




  0%|          | 0.00/200k [00:00<?, ?B/s]
100%|##########| 200k/200k [00:00<00:00, 759kB/s]
100%|##########| 200k/200k [00:00<00:00, 756kB/s]


### Extract the Dataset

The downloaded file is a ZIP file, so we extract the CSV file before loading it.

In [9]:
import zipfile

with zipfile.ZipFile("orders.csv.zip", "r") as zip_ref:
    zip_ref.extractall(".")

In [10]:
import os

print(os.listdir())

['.git', '.gitignore', '.ipynb_checkpoints', 'data', 'orders.csv', 'orders.csv.zip', 'orders_data_analysis.ipynb']


### Organize the Data Files

Keep the raw dataset inside a separate `data` folder to keep the project organized.

In [11]:
import os
import shutil

os.makedirs("data", exist_ok=True)

shutil.move("orders.csv", "data/orders.csv")
shutil.move("orders.csv.zip", "data/orders.csv.zip")

print(os.listdir("data"))

['orders.csv', 'orders.csv.zip']


## 3. Load and Explore the Raw Data

Load the CSV file into a pandas DataFrame and check the basic structure of the data.

In [12]:
import pandas as pd
df = pd.read_csv("data/orders.csv")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 9994
Columns: 16


In [13]:
df.head() # Preview the first few rows

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
0,1,2023-03-01,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Bookcases,FUR-BO-10001798,240,260,2,2
1,2,2023-08-15,Second Class,Consumer,United States,Henderson,Kentucky,42420,South,Furniture,Chairs,FUR-CH-10000454,600,730,3,3
2,3,2023-01-10,Second Class,Corporate,United States,Los Angeles,California,90036,West,Office Supplies,Labels,OFF-LA-10000240,10,10,2,5
3,4,2022-06-18,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Furniture,Tables,FUR-TA-10000577,780,960,5,2
4,5,2022-07-13,Standard Class,Consumer,United States,Fort Lauderdale,Florida,33311,South,Office Supplies,Storage,OFF-ST-10000760,20,20,2,5


In [14]:
df.info() # Check the data structure

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Order Id          9994 non-null   int64
 1   Order Date        9994 non-null   str  
 2   Ship Mode         9993 non-null   str  
 3   Segment           9994 non-null   str  
 4   Country           9994 non-null   str  
 5   City              9994 non-null   str  
 6   State             9994 non-null   str  
 7   Postal Code       9994 non-null   int64
 8   Region            9994 non-null   str  
 9   Category          9994 non-null   str  
 10  Sub Category      9994 non-null   str  
 11  Product Id        9994 non-null   str  
 12  cost price        9994 non-null   int64
 13  List Price        9994 non-null   int64
 14  Quantity          9994 non-null   int64
 15  Discount Percent  9994 non-null   int64
dtypes: int64(6), str(10)
memory usage: 1.2 MB


In [15]:
# Check for missing values
df.isnull().sum()

Order Id            0
Order Date          0
Ship Mode           1
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Category            0
Sub Category        0
Product Id          0
cost price          0
List Price          0
Quantity            0
Discount Percent    0
dtype: int64

In [16]:
# Check summary statistics
df.describe()

,Order Id,Postal Code,cost price,List Price,Quantity,Discount Percent
count,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000,9994.000000
mean,4997.500000,55190.379428,201.189714,229.756854,3.789574,3.484090
std,2885.163629,32063.693350,537.743203,623.245839,2.225110,1.114211
min,1.000000,1040.000000,0.000000,0.000000,1.000000,2.000000
25%,2499.250000,23223.000000,20.000000,20.000000,2.000000,2.000000
50%,4997.500000,56430.500000,50.000000,50.000000,3.000000,3.000000
75%,7495.750000,90008.000000,180.000000,210.000000,5.000000,4.000000
max,9994.000000,99301.000000,18110.000000,22640.000000,14.000000,5.000000


In [17]:
df.dtypes # Check the data types of each column

Order Id            int64
Order Date            str
Ship Mode             str
Segment               str
Country               str
City                  str
State                 str
Postal Code         int64
Region                str
Category              str
Sub Category          str
Product Id            str
cost price          int64
List Price          int64
Quantity            int64
Discount Percent    int64
dtype: object

In [18]:
df.isnull().sum() # Check for missing values

Order Id            0
Order Date          0
Ship Mode           1
Segment             0
Country             0
City                0
State               0
Postal Code         0
Region              0
Category            0
Sub Category        0
Product Id          0
cost price          0
List Price          0
Quantity            0
Discount Percent    0
dtype: int64

In [19]:
df[df["Ship Mode"].isnull()]

,Order Id,Order Date,Ship Mode,Segment,Country,City,State,Postal Code,Region,Category,Sub Category,Product Id,cost price,List Price,Quantity,Discount Percent
118,119,2023-07-19,NaN,Corporate,United States,Bristol,Tennessee,37620,South,Office Supplies,Binders,OFF-BI-10003650,140,160,1,5


In [20]:
df["Ship Mode"].unique()

<StringArray>
[  'Second Class', 'Standard Class',  'Not Available',        'unknown',
    'First Class',              nan,       'Same Day']
Length: 7, dtype: str

In [21]:
df["Ship Mode"].value_counts(dropna=False)

Ship Mode
Standard Class    5962
Second Class      1945
First Class       1538
Same Day           543
Not Available        4
unknown              1
NaN                  1
Name: count, dtype: int64

In [22]:
df.columns

Index(['Order Id', 'Order Date', 'Ship Mode', 'Segment', 'Country', 'City',
       'State', 'Postal Code', 'Region', 'Category', 'Sub Category',
       'Product Id', 'cost price', 'List Price', 'Quantity',
       'Discount Percent'],
      dtype='str')

In [23]:
# Standardize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")

In [24]:
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'cost_price', 'list_price', 'quantity',
       'discount_percent'],
      dtype='str')

In [25]:
# Calculate the discount amount
df["discount"] = df["list_price"] * df["discount_percent"] * 0.01

In [26]:
df[["list_price", "discount_percent", "discount"]].head()

,list_price,discount_percent,discount
0,260,2,5.2
1,730,3,21.9
2,10,5,0.5
3,960,2,19.2
4,20,5,1.0


In [27]:
# Calculate the sale price after discount
df["sale_price"] = df["list_price"] - df["discount"]

In [28]:
df[["list_price", "discount", "sale_price"]].head()

,list_price,discount,sale_price
0,260,5.2,254.8
1,730,21.9,708.1
2,10,0.5,9.5
3,960,19.2,940.8
4,20,1.0,19.0


In [29]:
df[["cost_price", "discount", "sale_price"]].head()

,cost_price,discount,sale_price
0,240,5.2,254.8
1,600,21.9,708.1
2,10,0.5,9.5
3,780,19.2,940.8
4,20,1.0,19.0


In [30]:
# Calculate profit
df["profit"] = df["sale_price"] - df["cost_price"]

In [31]:
df[["cost_price", "sale_price", "profit"]].head()

,cost_price,sale_price,profit
0,240,254.8,14.8
1,600,708.1,108.1
2,10,9.5,-0.5
3,780,940.8,160.8
4,20,19.0,-1.0


In [32]:
df["order_date"].dtype

<StringDtype(storage='python', na_value=nan)>

In [33]:
# Convert order date to datetime
df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="%Y-%m-%d"
)

In [34]:
df["order_date"].dtype

dtype('<M8[us]')

In [35]:
# Remove source pricing columns after calculations are complete
df = df.drop(
    columns=["list_price", "cost_price", "discount_percent"]
)

In [36]:
df.columns

Index(['order_id', 'order_date', 'ship_mode', 'segment', 'country', 'city',
       'state', 'postal_code', 'region', 'category', 'sub_category',
       'product_id', 'quantity', 'discount', 'sale_price', 'profit'],
      dtype='str')

In [37]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      9994 non-null   int64         
 1   order_date    9994 non-null   datetime64[us]
 2   ship_mode     9993 non-null   str           
 3   segment       9994 non-null   str           
 4   country       9994 non-null   str           
 5   city          9994 non-null   str           
 6   state         9994 non-null   str           
 7   postal_code   9994 non-null   int64         
 8   region        9994 non-null   str           
 9   category      9994 non-null   str           
 10  sub_category  9994 non-null   str           
 11  product_id    9994 non-null   str           
 12  quantity      9994 non-null   int64         
 13  discount      9994 non-null   float64       
 14  sale_price    9994 non-null   float64       
 15  profit        9994 non-null   float64       
dtyp

In [38]:
df.isnull().sum()

order_id        0
order_date      0
ship_mode       1
segment         0
country         0
city            0
state           0
postal_code     0
region          0
category        0
sub_category    0
product_id      0
quantity        0
discount        0
sale_price      0
profit          0
dtype: int64

In [39]:
# Standardize unknown shipping modes
df["ship_mode"] = df["ship_mode"].replace(
    ["Not Available", "unknown"],
    pd.NA
)

In [40]:
df["ship_mode"].value_counts(dropna=False)

ship_mode
Standard Class    5962
Second Class      1945
First Class       1538
Same Day           543
NaN                  6
Name: count, dtype: int64

In [41]:
df.isnull().sum()

order_id        0
order_date      0
ship_mode       6
segment         0
country         0
city            0
state           0
postal_code     0
region          0
category        0
sub_category    0
product_id      0
quantity        0
discount        0
sale_price      0
profit          0
dtype: int64

In [42]:
# Check the final dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      9994 non-null   int64         
 1   order_date    9994 non-null   datetime64[us]
 2   ship_mode     9988 non-null   str           
 3   segment       9994 non-null   str           
 4   country       9994 non-null   str           
 5   city          9994 non-null   str           
 6   state         9994 non-null   str           
 7   postal_code   9994 non-null   int64         
 8   region        9994 non-null   str           
 9   category      9994 non-null   str           
 10  sub_category  9994 non-null   str           
 11  product_id    9994 non-null   str           
 12  quantity      9994 non-null   int64         
 13  discount      9994 non-null   float64       
 14  sale_price    9994 non-null   float64       
 15  profit        9994 non-null   float64       
dtyp

In [43]:
# Check missing values
df.isnull().sum()

order_id        0
order_date      0
ship_mode       6
segment         0
country         0
city            0
state           0
postal_code     0
region          0
category        0
sub_category    0
product_id      0
quantity        0
discount        0
sale_price      0
profit          0
dtype: int64